In [15]:
import numpy as np
from scipy.optimize import minimize_scalar


def matrix_multiplication(matrix_a, matrix_b):
    if len(matrix_a[0]) != len(matrix_b):
        raise ValueError()

    result = [[0 for i in range(len(matrix_b[0]))] for j in range(len(matrix_a))]
    for i in range(len(matrix_a)):
        for j in range(len(matrix_b[0])):
            for k in range(len(matrix_b)):
                result[i][j] += matrix_a[i][k] * matrix_b[k][j]

    return result


def functions(a_1, a_2):
  list_of_coefficient_1 = list(map(float, a_1.split()))
  list_of_coefficient_2 = list(map(float, a_2.split()))

  #Исправленная версия нахождения экстремумов
  if list_of_coefficient_1[0] != 0:
    extremum_1 = minimize_scalar(lambda x: list_of_coefficient_1[0]*x**2 + list_of_coefficient_1[1]*x + list_of_coefficient_1[2])
    extremum_2 = minimize_scalar(lambda x: list_of_coefficient_2[0]*x**2 + list_of_coefficient_2[1]*x + list_of_coefficient_2[2])
    coordinates_of_extremum_1 = (extremum_1.x, extremum_1.fun)
    coordinates_of_extremum_2 = (extremum_2.x, extremum_2.fun)

  equation_coefficients = list()
  for i in range(len(list_of_coefficient_1)):
    equation_coefficients.append(list_of_coefficient_1[i] - list_of_coefficient_2[i])
  if all(item == 0 for item in equation_coefficients):
    return None
  if (equation_coefficients[0] == 0) and (equation_coefficients[1] == 0):
    return []
  if equation_coefficients[0] == 0:
    x_1 = - equation_coefficients[2] / equation_coefficients[1]
    y_1 = list_of_coefficient_1[0]*x_1**2 + list_of_coefficient_1[1]*x_1 + list_of_coefficient_1[2]
    return [(x_1, y_1)]
  D = equation_coefficients[1]**2 - 4*equation_coefficients[0]*equation_coefficients[2]
  if D < 0:
    return []
  if D == 0:
    x_1 = -equation_coefficients[1] / (2 * equation_coefficients[0])
    y_1 = list_of_coefficient_1[0]*x_1**2 + list_of_coefficient_1[1]*x_1 + list_of_coefficient_1[2]
    return [(x_1, y_1)]
  x_1 = (-equation_coefficients[1] + np.sqrt(D)) / (2 * equation_coefficients[0])
  x_2 = (-equation_coefficients[1] - np.sqrt(D)) / (2 * equation_coefficients[0])
  y_1 = list_of_coefficient_1[0]*x_1**2 + list_of_coefficient_1[1]*x_1 + list_of_coefficient_1[2]
  y_2 = list_of_coefficient_1[0]*x_2**2 + list_of_coefficient_1[1]*x_2 + list_of_coefficient_1[2]
  return [(x_1, y_1), (x_2, y_2)]


def skew(x):
    n = len(x)
    mean = sum(x) / n
    variance = sum((i - mean) ** 2 for i in x) / n
    std_dev = np.sqrt(variance)
    skew = sum((i - mean) ** 3 for i in x) / (n * std_dev**3)

    return round(skew, 2)


def kurtosis(x):
    n = len(x)
    mean = sum(x) / n
    variance = sum((i - mean) ** 2 for i in x) / n
    std_dev = np.sqrt(variance)
    kurt = sum((i - mean) ** 4 for i in x) / (n * std_dev**4) - 3

    return round(kurt, 2)

In [6]:
import random
import unittest
import scipy.stats


class MATHTestCase(unittest.TestCase):
    def test_matrix_multiplication(self):
        a1 = [[1, 2, 3],
             [4, 5, 6]]
        b1 = [[7, 8],
             [9, 10],
             [11, 12]]
        self.assertEqual(matrix_multiplication(a1, b1), [[58, 64],[139, 154]])

        a2 = [[1, 2],
             [3, 4]]
        b2 = [[1, 2]]
        with self.assertRaises(ValueError):
            matrix_multiplication(a2, b2)

        a3 = [[1, 2, 3],
             [4, 5, 6],
             [7, 8, 9]]
        b3 = [[1, 0, 0],
             [0, 1, 0],
             [0, 0, 1]]
        self.assertEqual(matrix_multiplication(a3, b3), a3)

        a4 = [[1, 2, 3],
             [4, 5, 6]]
        b4 =[[0, 0],
             [0, 0],
             [0, 0]]
        self.assertEqual(matrix_multiplication(a4, b4), [[0, 0], [0, 0]])

        a5 = [[1, 2, 3]]
        b5 = [[4],
             [5],
             [6]]
        self.assertEqual(matrix_multiplication(a5, b5), [[32]])

        a6 = [[3]]
        b6 = [[4]]
        self.assertEqual(matrix_multiplication(a6, b6), [[12]])


    def test_functions(self):
        coeffs1 = "1 0 -4"
        coeffs2 = "1 -2 0"
        self.assertEqual(functions(coeffs1, coeffs2), [(2, 0)])

        coeffs3 = "1 0 4"
        coeffs4 = "1 0 1"
        self.assertEqual(functions(coeffs3, coeffs4), [])

        coeffs5 = "1 2 1"
        coeffs6 = "1 2 1"
        self.assertIsNone(functions(coeffs5, coeffs6))

        coeffs7 = "1 2 3"
        coeffs8 = "1 2 1"
        self.assertEqual(functions(coeffs7, coeffs8), [])

        coeffs9 = "0 2 -1"
        coeffs10 = "1 -4 4"
        self.assertEqual(functions(coeffs9, coeffs10), [(1, 1), (5, 9)])


    def test_skew(self):
        x1 = [2,3,5,7,8]
        x2 = [2,3,2,5,7,2,2,8]
        self.assertEqual(skew(x1), round(scipy.stats.skew(x1), 2))
        self.assertEqual(skew(x2), round(scipy.stats.skew(x2), 2))

        random.seed(100)
        random_floats = [random.random() for _ in range(10000)]
        random_integers = [random.randint(1, 99) for _ in range(10000)]
        self.assertEqual(skew(random_floats), round(scipy.stats.skew(random_floats), 2))
        self.assertEqual(skew(random_integers), round(scipy.stats.skew(random_integers), 2))


    def test_kurtosis(self):
        x1 = [2,3,5,7,8]
        x2 = [2,3,2,5,7,2,2,8]
        self.assertEqual(kurtosis(x1), round(scipy.stats.kurtosis(x1), 2))
        self.assertEqual(kurtosis(x2), round(scipy.stats.kurtosis(x2), 2))

        random.seed(100)
        random_floats = [random.random() for _ in range(10000)]
        random_integers = [random.randint(1, 99) for _ in range(10000)]
        self.assertEqual(kurtosis(random_floats), round(scipy.stats.kurtosis(random_floats), 2))
        self.assertEqual(kurtosis(random_integers), round(scipy.stats.kurtosis(random_integers), 2))

In [7]:
tests = MATHTestCase()

In [16]:
tests.test_functions()